# Quantum Chemistry with ORCA

> **FIXME for the final version**: Remove the energy printing code blocks from butanol exercise

## Introduction

In this notebook, we will carry out quantum chemical calculations on organic and inorganic molecules. 
- We will use the ORCA program package developed at the Max-Planck-Institut für Kohlenforschung.
- ORCA is a general-purpose quantum chemistry package initiated by Frank Neese in 1999.
- ORCA is free for academic use and features a variety of quantum chemical methods.
- We will be using Density Functional Theory (DFT) and Gaussian-Type basis sets.

## Learning outcomes

After completing this notebook, you will be able to:
- Understand the key parts of ORCA input files for molecular systems.
- Optimize molecular structures with ORCA.
- Compare total energies of molecular isomers to find out the lowest-energy isomer.
- Calculate and interpret infrared vibrational spectra for molecules.

## Instructions for using the notebook
- The notebook is composed of cells and the blue highlight in the left side of the notebook shows which cell is currently active. 
- You can run the active cell by clicking the "play" button from the toolbar above ("Run this cell and advance", keyboard shortcut is Shift + Enter).
- The notebook includes instruction cells like this cell and gray cells that contain Python code.
- When you run a cell and see the symbol `In [*]` in the top left corner of the cell, the notebook is executing code. Please wait until the symbol changes to a number like `[1]`.
- Run the cells from the top to the bottom.
- The files generated during the exercise are visible in the **File browser** on the left. Text files (.inp, .xyz, and .out extension) can be opened for viewing by double-clicking them.
- If you get lost in the notebook, you can view the Table of Contents from View -> Table of Contents. View -> File Browser brings back the File Browser. These are also available as icons on the vertical icon bar on the left.
- If something goes totally wrong, you can open the Kernel menu and choose "Restart Kernel and Clear Outputs of All Cells". This will reset the state of the notebook and you can start from the beginning.

## Part 0. Setting up the calculation environment for ORCA

Run the next code block to setup the calculation environment for ORCA.

In [ ]:
import sys
sys.path.append('../tools')
from qctools import load_xyz, save_xyz, load_molecule_pubchem, load_xyz_as_traj, show_molecule

# The total electronic energies in ORCA are in Hartree units.
# We use this conversion factor throughout the notebook to convert Hartrees to kJ/mol.
HARTREE_TO_KJMOL = 2625.5

# A short wrapper function to run ORCA jobs. 
# jobdir and jobname.inp must exist.
def run_orca(jobdir: str = None, jobname: str = None):
    if jobdir is None:
        print("jobdir cannot be None")
    elif jobname is None:
        print("jobname cannot be None")
    else:
        print(f"Running ORCA input file {jobdir}/{jobname}.inp ...")
        !cd {jobdir} ; $$ORCA_HOME/orca {jobname}.inp > {jobname}.out
        !grep -E "TERMINATED|TOTAL RUN TIME" {jobdir}/{jobname}.out
        # Technical note:
        # In a Jupyter notebook, $$ORCA_HOME corresponds to environment variable $ORCA_HOME. 
        # This is the directory where we have set up the ORCA binaries.

## Part 1. ORCA walkthrough for a simple molecule: H<sub>2</sub>O
Let's get familiar with ORCA by using a gas-phase H<sub>2</sub>O molecule as an example.

### Creating subdirectories
ORCA creates a large number of files during the execution. In this notebook, all ORCA input and output files are stored inside subdirectories. 

Run the following code block to create a subdirectory `h2o` for our first job.

In [ ]:
!mkdir h2o

`!mkdir` means that the notebook calls the Unix shell command `mkdir` to create a directory. We could also create directories with Python but in this case it is more convenient to work directly with shell commands.

> The newly created subdirectory `h2o` should soon appear in the file browser on the left.

### Writing files and visualizing molecular structures

Let's start by visualizing a molecular structure using NGLview tool. We first need to create a molecular structure file. 
- We can write files in notebooks with the `%%writefile` command (a so-called IPython magic command available in notebooks).
- `%%writefile h2o/h2o_dist.xyz` means that we will create a file called `h2o_dist.xyz` in the subdirectory `h2o` that we just created above.
- The file format is so-called XYZ file. This is a standard human-readable file format in quantum chemistry. It contains the cartesian coordinates of each atom in format `symbol X Y Z`).

Run the following code block to create an XYZ file with a distorted water molecular (H-O-H angle of 60 degrees, O-H distances of 1.2 Å).

In [ ]:
%%writefile h2o/h2o_dist.xyz
3
Distorted water molecule
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361

> You can use the file browser on the left to check that the file `h2o_dist.xyz` indeed appeared inside the `h2o` subdirectory (double-click the folder to open it). You can return to the original working directory by clicking `basic` in the directory path `/ ... / basic / h2o` in the top of the file browser.

Next, run the following code block to show a visualization of the distorted water molecule using NGLview.

In [ ]:
h2o_dist_view = load_xyz('h2o/h2o_dist.xyz')
show_molecule(h2o_dist_view)

### Interacting with the molecular structure
You can interact with the molecular structure in the above view as follows:
- Rotate: hold the left mouse button and move the mouse.
- Zoom: mouse scrollwheel.
- Translate: hold the right mouse button and move the mouse.
  
Next, we will optimize the distorted structure to a reasonable minimum. 

### Structure optimization of H<sub>2</sub>O

In quantum chemical molecular modelling, the first task is usually the structural optimization of the studied system. This means that we will find the lowest-energy three-dimensional structure for the molecule.

Standard structure optimization methods can find local minima on the potential energy surface. This means that the optimization algorithm does not search all possible conformations of the molecule. We will have a look into conformation analysis later.

Let's optimize the structure of water molecule with Density Functional Theory (DFT). We will use the DFT-PBE functional, which is the most commonly used GGA-level DFT functional. We will look into more accurate DFT methods later, but for typical main group compounds DFT-PBE performs reasonably well. 

Let's start by writing an ORCA input file. The input file shown in the code block below consists of following items:
- The first line in the code block is the `%%writefile` command that writes the following lines into a file `h2o_opt.inp` located in subdirectory `h2o` created above. The convention is that ORCA input files have file extension `.inp`.
- The line starting with `!` is so-called ORCA simple input line. Here we define the DFT-PBE method (`PBE`) and the one-electron basis set (`DEF2-TZVP`). PBE belongs to the *Generalized Gradient Approximation* (GGA) family of DFT functionals. def2-TZVP is a triple-zeta-valence + polarization level basis set which is typically sufficient for all routine DFT calculations.
- **IMPORTANT**: It is good to be aware that ORCA uses by default so-called *resolution of the identity* (RI) approximation for DFT calculations. The RI approximation speeds up the calculations significantly without any practical loss in accuracy. ORCA also automatically selects the so-called auxilliary basis set for this approximation (def2/J).
- The `Opt` keyword means that we want to optimize the molecular structure.
- The simple input line is followed by the XYZ coordinates of the water molecule. The line `xyz 0 1` tells ORCA that we will give the molecular structure in XYZ format, the charge of the molecule is 0 and the spin multiplicity is 1 (singlet, no unpaired electrons).
- The calculation will use only one CPU core because the system is so small. Later, we will add more CPUs (computing power).

Run the next code block to create the input file.

In [ ]:
%%writefile h2o/h2o_opt.inp
!PBE DEF2-TZVP Opt
* xyz 0 1
 O                  0.00000000    0.00000000   -0.16627688
 H                  0.00000000   -0.60000000    0.87295361
 H                 -0.00000000    0.60000000    0.87295361
*

On a general level, the structural optimization will proceed as follows:

- ORCA calculates the electronic structure (molecular orbitals) of the initial structure. This also yields the electronic total energy of the system.
- ORCA calculates the forces affecting the atoms (first derivatives of the total energy with respect to atomic coordinates).
- ORCA tries to displace the atoms in such way that the forces affecting the atoms become smaller. The goal is to reach a local minimum structure where all forces are zero (in practice, close to zero).
- The above three steps are repeated until the forces affecting the atoms are below the default convergence criteria in ORCA. These are relatively strict, and can be changed by the user.
- After the convergence criteria have been reached, the structure optimization is complete.

Run the next code block to run the structure optimization. The job should take less than 30 seconds to finish.

In [ ]:
run_orca(jobdir = "h2o", jobname = "h2o_opt")

Let's load all intermediate structures of the optimization to see how it proceeded (this is often called the "trajectory"). 

Run the following code block, which loads the optimization trajectory. After loading, the view shows the initial structure. Move your mouse  over the view and press the ▶ button to see how the optimization changed the molecular structure.

In [ ]:
h2o_opt_view = load_xyz_as_traj('h2o/h2o_opt_trj.xyz')
show_molecule(h2o_opt_view)

### Energy changes during the structural optimization of H<sub>2</sub>O

Let's see how the total electronic energy of the system changed during the optimization. The next code block will read the total electronic energies and plot them with Matplotlib. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the electronic total energies from the ORCA output
energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o/h2o_opt.out | awk '{print $NF}'

energies = np.asarray(energies_raw, dtype=float)
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, energies, marker = 'o', color = 'black', label = 'Total energy')
plt.xlabel('Optimization step')
plt.ylabel('Total energy (Hartree)')
plt.xlim(1, len(steps) + 1)
plt.ylim(np.min(energies)*1.0001, np.max(energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(energies), max(energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows how the total energy decreases and then no longer changes. In chemistry, we are usually more interested in relative energies. Let's see how the same plot looks when we set the energy of the optimized structure as zero and calculate the relative energies of the intermediate structures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract the electronic total energies from the ORCA output
energies_raw = !grep "FINAL SINGLE POINT ENERGY" h2o/h2o_opt.out | awk '{print $NF}'
energies = np.asarray(energies_raw, dtype=float)

# Set the minimum energy as the zero level and calculate relative energies.
# The total energies in ORCA are in Hartree units. 
# We convert the relative energies to kJ/mol.
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Optimization step')
plt.ylabel('Relative energy (kJ/mol)')
plt.xlim(1, len(steps) + 1)
plt.ylim(0, np.max(rel_energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(rel_energies), max(rel_energies), 6))
plt.legend(loc = 'upper right')
plt.title('Optimization of H$_2$O')
plt.show()

The plot above shows that the starting structure was indeed a high-energy structure (224 kJ/mol higher in energy compared to the final minimum).

### Harmonic frequency calculation for H<sub>2</sub>O

The structure optimization of the water molecule finished because all forces affecting the atoms are close to zero. This means that we have reached a stationary point at the potential energy surface.

Next, we can run a so-called frequency calculation to check that we have reached a true local minimum. The frequencies are typically calculated within the harmonic approximation. This means that the vibrational modes are described as harmonic oscillators that are independent of each other. It is also possible to study anharmonic frequencies where the vibrational modes are coupled to each other but this is out of the scope of the current tutorial.

In a frequency calculation, we are calculating the second derivatives of the total energy with respect to atomic coordinates. This yields so-called Hessian matrix (in short, Hessian) which describes the local curvature of the potential energy surface:
- If all eigenvalues of the Hessian matrix are positive, structure is a true local minimum. All vibrational frequencies are positive.
- If the Hessian has one negative eigenvalue, the structure is a transition state. One vibrational frequency has an imaginary value (often denoted as negative frequency due to technical reasons)
- If the Hessian has several negative eigenvalues, the structure is a saddle point. There are several imaginary vibrational frequencies.

Let's write an ORCA input file for the frequency calculation. The input file shown in the next code block has some changes compared to the structure optimization input:
- The keyword `Opt` has been replaced with keyword `Freq`.
- The line `xyzfile 0 1 h2o_opt.xyz` tells ORCA to read the optimized structure from the file `h2o_opt.xyz`. Charge is 0 and spin multiplicity is 1 similar to the optimization input above.

Run the next code block to write the input file for the frequency calculation.

In [ ]:
%%writefile h2o/h2o_fq.inp
!PBE DEF2-TZVP Freq
* xyzfile 0 1 h2o_opt.xyz

Run the next code block to run the frequency calculation. The job should take less than 30 seconds to finish.

In [ ]:
run_orca(jobdir = "h2o", jobname = "h2o_fq")

The next code block will extract the calculated vibrational frequencies from the output file using the `sed` utility.

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' h2o/h2o_fq.out | head -n -3

There are in principle 9 vibrational frequencies listed (note that indexing starts from 0). However, the first six models are zero frequencies that correspond to translations and rotations. For a non-linear molecule with *N* atoms, there are 3*N*−6 nonzero vibrational modes. 

### Visualizing the harmonic frequencies of H<sub>2</sub>O

We can plot the three nonzero modes in `xyz` trajectory format with the `orca_pltvib` command:

In [ ]:
!cd h2o ; orca_pltvib h2o_fq.hess 6 7 8

After this, the trajectory files located in the `h2o` subdirectory can be visualized to show the vibrational frequencies.

After running the next code block, move your mouse over the view, and press the third button in the toolbar that appears ("repeat"). Then, press play to see the vibrational frequency.

In [ ]:
# Visualize the vibrational frequency 6 at 1583.29 cm**-1
h2o_freq_view = load_xyz_as_traj('h2o/h2o_fq.hess.v006.xyz')
show_molecule(h2o_freq_view)

> #### Exercise
> Modify the **trajectory file name** in the previous previous code and re-run the code block to visualize the two other vibrational frequencies of water.

### Infrared spectrum of H<sub>2</sub>O in the gas phase

Finally, we can plot the IR spectrum of water molecule in the gas phase. IR intensity of a vibrational mode depends on the change of dipole moment during the vibration. ORCA calculates the IR intensities by default during a harmonic frequency calculation.

First, generate the IR spectrum plot with the `orca_mapspc` tool:

In [ ]:
!cd h2o ; orca_mapspc h2o_fq.hess IR -w32

Parameter `-w32` determines the linewidth (Gaussian shape, Full width at half maximum = 32 cm<sup>−1</sup>). Next, plot the spectrum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the raw data
ir_data = np.loadtxt('h2o/h2o_fq.hess.ir.dat')

frequencies = ir_data[:, 0] # Wavenumbers in the first column
transmittances = (ir_data[:, 1] - 999) * 100 # Transmittances in the second column
plt.plot(frequencies, transmittances, '-', color = 'red', label = 'IR')
plt.xlabel('Frequency (cm$^{-1}$)')
plt.ylabel('Transmittance (%)')
plt.xlim(max(frequencies), min(frequencies), )
plt.ylim(min(transmittances) * 0.999, max(transmittances) * 1.001)
plt.xticks(np.linspace(500, 4000, 8))
plt.yticks([])
plt.title('IR spectrum of H$_2$O')
plt.show()

Note that the IR spectrum above is for **one H<sub>2</sub>O molecule in the gas phase**. It is very different from the IR spectrum of liquid water where intermolecular interactions change the picture significantly.

Final note: It is possible to include both `Opt` and `Freq` keywords in the same input. In this case, Orca first optimizes the strucure and then calculates the harmonic frequencies. We will use this feature in the next part.

## Part 2. Complete workflow for an inorganic 5*d* metal compound: WF<sub>6</sub>

Tungsten hexafluoride WF<sub>6</sub> is used as a precursor molecule to form tungsten thin-films in semiconductor industry. Let's study the structure and vibrational properties of WF<sub>6</sub>.

Let's first create a new subdirectory `wf6`:

In [ ]:
!mkdir wf6

When investigating *d*-metals, it is important to pay attention on the DFT exchange-correlation (XC) functional. The DFT-PBE method we used above for water may not be accurate enough. In this example, we will be using the hybrid **DFT-PBE0** functional which includes 25% of exact exchange (Hartree-Fock exchange) in the XC functional. 

**IMPORTANT**: It is good to be aware that for hybrid functionals ORCA uses by default so-called RIJCOSX approximation which combines RI approximation with semi-numerical *chain-of-spheres* approximation for the exchaxt exchange. This speeds up hybrid DFT calculations significantly without significant losses in accuracy.

We will be using the same def2-TZVP basis set as in the example above. However, in this case, 60 core electrons of tungsten are replaced by an effective core potential (ECP). This leaves 14 electrons on each tungsten atom. The ECP takes care of scalar relativistic effects and also decreases the computational cost as 60 less electrons need to be treated explicitly.

### Structural optimization and frequency calculation of WF<sub>6</sub>
The first step is to write the input file. Usually it is convenient to carry out both structure optimization and harmonic frequency calculation in the same job. This can be done by including both `Opt` and `Freq` keywords. 

WF<sub>6</sub> has a lot more electrons compared to H<sub>2</sub>O and we are using a hybrid DFT method that is computationally more demanding than the GGA-type PBE functional we used before. Therefore, we will this time run the job in parallel with four processors (keyword `PAL4`).

In [ ]:
%%writefile wf6/wf6.inp
!PBE0 DEF2-TZVP Opt Freq
!PAL4
* xyz 0 1
 W                 -0.13573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664    1.87644611
 F                  1.74426503    0.20359664   -0.00355389
 F                 -2.01573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664   -1.88355389
 F                 -0.13573497    2.08359664   -0.00355389
 F                 -0.13573497   -1.67640336   -0.00355389
*

Run the job (will take about 6 to 8 minutes):

In [ ]:
run_orca(jobdir = "wf6", jobname = "wf6")

Let's see how the total energy changed during the optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

energies_raw = !grep "FINAL SINGLE POINT ENERGY" wf6/wf6.out | awk '{print $NF}'

energies = np.asarray(energies_raw, dtype=float)
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
steps = np.arange(1, len(energies_raw) + 1)

plt.plot(steps, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Optimization step')
plt.ylabel('Relative energy (kJ/mol)')
plt.xlim(1, len(steps) + 1)
plt.ylim(0, np.max(rel_energies))
plt.xticks(np.arange(1, len(steps) + 1, 1))
plt.yticks(np.linspace(np.min(rel_energies), max(rel_energies), 6))
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
plt.legend(loc = 'upper right')
plt.title('Optimization of WF$_6$')
plt.show()

Let's visualize the optimization trajectory

In [ ]:
wf6_view = load_xyz_as_traj('wf6/wf6_trj.xyz')
show_molecule(wf6_view)

Extract the calculated vibrational frequencies from the output file:

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' wf6/wf6.out | head -n -3

> #### Question
>
> Did the structure optimization reach a true local minimum? How can you tell?
> Answer in the raw text box below.

Plot all vibrational modes in `xyz` trajectory format with the `orca_pltvib` command:

In [ ]:
!cd wf6 ; orca_pltvib wf6.hess all

The vibrational frequencies can be visualized as in the example above. Let's first visualize the lowest-frequency vibrational mode 6:

In [ ]:
wf6_freq_view = load_xyz_as_traj('wf6/wf6.hess.v006.xyz')
show_molecule(wf6_freq_view)

> #### Question
> What is the wavenumber (frequency) of the fully symmetric W–F stretching vibration (all W–F bonds are stretching in the same phase)? Change the name of the trajectory file name in the code block above and visualize the vibrations to find the answer.

### Molecular orbitals of WF<sub>6</sub>

 As a result of the DFT-PBE0 calculation, we have obtained the molecular orbitals (MOs) of WF<sub>6</sub> (strictly speaking, we obtained the Kohn-Sham orbitals because we are using Kohn-Sham DFT, but in practice we can call them MOs).

Run the next code block to print the energies of the individual MOs. ORCA prints the MO energies during the first step of the structural optimization and after the optimization has converged. If you are ever looking for molecular orbitals energies from an optimization output, be sure to take the values in the end of the file (after `FINAL ENERGY EVALUATION`).

In [ ]:
!sed -n -e '/FINAL ENERGY EVALUATION/,/Only the first 10/p' wf6/wf6.out | sed -n -e '/ORBITAL ENERGIES/,/Only the first 10/p'

The indexing of the MOs starts from 0. The orbitals with OCC = 2 are occupied while those with OCC = 0 are virtual (empty) orbitals. The highest occupied MO (HOMO) is orbital 33. Because the indexing of the orbitals start from zero, we have (33 + 1) * 2 = 68 electrons. We can double check this by printing the number of electrons from the ORCA output: 

In [ ]:
!grep -m 1 "Number of Electrons" wf6/wf6.out

MO isosurfaces (plots) can be created with command `orca_plot`. The command reads the wavefunction from the file `wf6.gbw`. It is possible to run `orca_plot` interactively with `orca_plot gbwfile -i`. However, we will run the command with an input file that lists the orbitals that we want to plot.

We will be writing an input file like this (the example includes comments for clarity, they will not be included in the final input)
```
%%writefile wf6/moplot
2   # Tell orca_plot to read MO number
33  # MO number
11  # Plot the MO (writes a so-called cube file)
12  # Exit orca_plot
```
Run the next code block to save an input that generates cube files for MOs 33, 30, and 21.

In [ ]:
%%writefile wf6/moplot
2
33
11
2
30
11
2
21
11
12

Run `orca_plot`:

In [ ]:
!cd wf6 ; orca_plot wf6.gbw -i < moplot > /dev/null

Visualize the three MOs (33, 30, 21) one at a time by setting the variable `CUBEFILE` in the following code block and run it. The molecule is rather far when it loads, so you need to zoom in (with mouse wheel).

In [ ]:
# Set cube that you want to plot: mo33a, mo30a, or mo21a
CUBEFILE = "wf6/wf6.mo33a.cube"
# You can adjust ISOLEVEL between 0.01 and 0.08 to see how the extent of the orbital changes.
# The larger the number, the tighter the isosurface value cutoff for plotting.
ISOLEVEL = 0.03
# ----- No need to change anything below -----
wf6_geom = load_xyz('wf6/wf6.xyz')
moview = show_molecule(wf6_geom)
moview.add_component(CUBEFILE)
moview.component_1.clear()
moview.component_1.add_surface(opacity=0.5, color='blue', isolevelType="value", isolevel=ISOLEVEL)
moview.component_1.add_surface(opacity=0.5, color='red',  isolevelType="value", isolevel=-ISOLEVEL)
moview

The MO plots illustrate that these so-called canonical MOs are typically rather delocalised. It is possible to localize the molecular orbitals to get localized orbitals that are typically easier to connect to chemical bonding concepts typically discussed in textbooks.

> #### Question
>
> In which of the three MOs the tungsten atom appears to have the largest contribution?

## Part 3. Structural isomers of butanol

[Butanol](https://en.wikipedia.org/wiki/Butanol) C<sub>4</sub>H<sub>9</sub>OH is an alcohol with four different structural isomers: 1-butanol, 2-butanol, isobutanol, and tert-butanol.

The goal of this exercise is to find out the energy ordering of the structural isomers. In other words, we would like to know which isomer has the lowest total energy and which one has the highest. 

> Regarding the meaning of "the lowest": the total electronic energies are negative numbers and when comparing for example two values −134.56 Hartree and −133.22 Hartree, the value -134.56 would be the "lowest".

Let's first create a new subdirectory for this exercise by running the following code block:

In [ ]:
!mkdir butanol

Next, we need to get the initial structural models for the butanol isomers.

### Loading the butanol isomers from PubChem

PubChem is one of the largest openly available molecular databases in the internet. Let's load the initial structural models of the butanol isomers from there.

The following code block loads the 3D coordinates of the four butanol isomers from PubChem and visualizes them. Loading the molecules from the database might take a while (up to 10 seconds).

In [ ]:
# Initialize a dictionary that will contain both the isomer name and its total energy (in Hartree)
isomers = {'1-butanol': 0, '2-butanol': 0, 'isobutanol': 0, 'tert-butanol': 0}

# The code below calls load_molecule_pubchem for each isomer. 
# This function also creates an XYZ file for each isomer.
from ipywidgets import TwoByTwoLayout, Label, VBox, Layout
views = []
for isomer in isomers.keys():
    !mkdir butanol/{isomer}
    mol = load_molecule_pubchem(isomer, xyzfile = f"butanol/{isomer}/{isomer}_pubchem.xyz")
    
    # The rest is for visualization
    view = show_molecule(mol)
    view.layout.width = 'auto'
    label = Label(isomer)
    container = VBox([label, view], layout = Layout(padding='0px 10px 10px 0px'))
    views.append(container)

# Display the isomers in a two-by-two grid
grid = TwoByTwoLayout(top_left=views[0], top_right=views[1], bottom_left=views[2], bottom_right=views[3])
display(grid)

Now we can create ORCA inputs for all four isomers. This time we use the Python code block below to create the input files.

We will be using the hybrid PBE0 functional. Instead of the full def2-TZVP basis set, let's use a slightly smaller basis set def2-TZVP(-f), from which the f-type polarization functions have been removed. 

> As a technical detail, we will use the so-called RIJK approximation instead of the RIJCOSX approximation that is the default for hybrid DFT. In the case of small molecules such as butanol, RIJK is typically faster than RIJCOSX (RIJCOSX would take twice as much time in this case). Keywords `RIJK` and `def2/JK` are related to this choice. Note that if you need to calculate harmonic frequencies, you probably should not use RIJK (analytical frequency calculation is not available, that is, things will get more complicated).

We will run the jobs in parallel to save time (although for such small molecules the parallelization speedup is somewhat modest).

In [ ]:
for isomer in isomers.keys():
    try:
        filepath = f"butanol/{isomer}/{isomer}.inp"
        with open(filepath, "w") as orcainput:
            orcainput.write("!PBE0 DEF2-TZVP(-f) RIJK def2/JK Opt\n"
                            "!PAL4\n"
                            f"* xyzfile 0 1 {isomer}_pubchem.xyz\n")
            print(f"Created ORCA input file {filepath}")
    except IOError:
        print(f"Could not create file {filepath}!")
        break

Next, run the jobs. Each of the four isomer optimizations will take about 1-2 minutes.

In [ ]:
for isomer in isomers.keys():
    run_orca(jobdir = f"butanol/{isomer}", jobname = isomer)
    print("----------------------------------------------------------")

### Energy comparison of the structural isomers

We can now use the total energies of the structural isomers to calculate their relative energies.

Find out the total energy of each isomer and fill the energies in the following code block. Hint: you can find the total energy of the opimized structure as the very last `FINAL SINGLE POINT ENERGY` in the main output file (`jobname.out`) or from the file `jobname.engrad`.

In [ ]:
# TO BE REMOVED FROM THE FINAL VERSION
for isomer in isomers.keys():
    output = f"butanol/{isomer}/{isomer}.out"
    print(f"File: {output}") 
    !grep "FINAL SINGLE POINT ENERGY" {output} | tail -1
    print("")

In [ ]:
# Fill in total energies (in Hartree units)
isomers["1-butanol"] = 0
isomers["2-butanol"] = 0
isomers["isobutanol"] = 0
isomers["tert-butanol"] = 0
###################################
OK = True
lowest_E = 0
lowest_isomer = ""
for isomer, energy in isomers.items():
    if energy == 0:
        print(f"Energy value missing for isomer: {isomer}")
        OK = False
        break
    else:
        if energy < lowest_E:
            lowest_E = energy
            lowest_isomer = isomer
if OK:        
    print(f"The isomer {lowest_isomer} has the lowest total energy: {lowest_E} Hartree\n")

    print("Relative energies of the isomers in kJ/mol units:")
    for isomer, energy in isomers.items():
        E_rel = (energy - lowest_E) * HARTREE_TO_KJMOL
        print(f"{isomer:13s} {E_rel:3.1f} kJ/mol")

> #### Question
>
> Which structural isomer has the lowest energy? Why?

It's important to note that the energy comparison here applies for individual gas-phase molecules at 0 K. Energy comparison at the room temperature would require consideration of thermodynamic quantities and condensed phase effects (all butanol isomers are liquid at room temperature). 

However, let's compare the present 0 K gas-phase results with existing condensed phase thermodynamic data. The [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) lists the following condensed phase heats of formation for the butanol isomers (Δ$_f$H°<sub>liquid</sub>):

<div style="display: inline-block; text-align: left">
    
| Isomer       | Δ$_f$H° (kJ/mol) |
|--------------|------|
| 1-butanol    | -327 |
| 2-butanol    | -343 |
| isobutanol   | -335 |
| tert-butanol | -359 |

</div>

*(note that the experimental data includes standard deviations but we are omitting them here)*

> #### Question
> How do these experimental values compare with the gas-phase DFT-PBE0 values?

### Dispersion corrections

Note that none of the DFT functionals used above can properly describe weak dispersive (van der Waals) interactions. This is the case with practically all standard DFT functionals. Dispersion interactions are particularly important for describing weak intermolecular interactions but molecules have intramolecular dispersion interactions, too.

We can take dispersion interactions into account by using empirical dispersion corrections. This is most commonly done by means of D3 or D4 dispersion corrections developed by Grimme and coworkers. 

Switching on the dispersion corrections in ORCA is very simple: just add `D3` or `D4` in the simple input line.

Run the following code block to create new input files for all butanol isomers, this time with the D3 correction switched on.

In [ ]:
for isomer in isomers.keys():
    try:
        filepath = f"butanol/{isomer}/{isomer}_D3.inp"
        with open(filepath, "w") as orcainput:
            orcainput.write("!PBE0 D3 DEF2-TZVP(-f) RIJK def2/JK Opt\n"
                            "!PAL4\n"
                            f"* xyzfile 0 1 {isomer}_pubchem.xyz\n")
            print(f"Created ORCA input file {filepath}")
    except IOError:
        print(f"Could not create file {filepath}!")
        break

Run the following code block to optimize the butanol isomers with the D3 dispersion correction (DFT-PBE0-D3). Each of the four optimizations will take about 1-2 minutes.

In [ ]:
for isomer in isomers.keys():
    run_orca(jobdir = f"butanol/{isomer}", jobname = f"{isomer}_D3")
    print("-----------------------------------------------")

Similar to the energy comparison above, find out the total energy of each isomer (now from the output files with D3 correction) and fill the energies in the energy comparison code block below.

In [ ]:
# TO BE REMOVED FROM THE FINAL VERSION
for isomer in isomers.keys():
    output = f"butanol/{isomer}/{isomer}_D3.out"
    print(f"File: {output}") 
    !grep "FINAL SINGLE POINT ENERGY" {output} | tail -1
    print("")

In [ ]:
# Fill in total energies with D3 correction (in Hartree units)
isomers["1-butanol"] = 0
isomers["2-butanol"] = 0
isomers["isobutanol"] = 0
isomers["tert-butanol"] = 0
###################################
OK = True
lowest_E = 0
lowest_isomer = ""
for isomer, energy in isomers.items():
    if energy == 0:
        print(f"Energy value missing for isomer: {isomer}")
        OK = False
        break
    else:
        if energy < lowest_E:
            lowest_E = energy
            lowest_isomer = isomer
if OK:        
    print(f"The isomer {lowest_isomer} has the lowest total energy: {lowest_E} Hartree\n")

    print("Relative energies of the isomers in kJ/mol units:")
    for isomer, energy in isomers.items():
        E_rel = (energy - lowest_E) * HARTREE_TO_KJMOL
        print(f"{isomer:13s} {E_rel:3.1f} kJ/mol")

> #### Question
> Did the energy ordering of the isomers change? How did the relative energies change? Did the agreement with the experimental data improve or worsen?

## Part 4. Conformational analysis of ethylene glycol

In the previous example on butanol isomers we did not carry out any conformational analysis on the different structural isomers. At the room temperature, any molecule in the liquid or gas phase might possess several conformers. 

ORCA has a powerful algorithm called GOAT (*global geometry optimization and ensemble generator*) that can be used to find out the lowest-energy conformers of the studied molecule.

Let's use ethylene glycol (ethane-1,2-diol, (CH<sub>2</sub>OH)<sub>2</sub>) as our example system. Ethylene glycol is used for example as an antifreeze in automotive engines (*pakkasneste* in Finnish).

The figure below shows a 2D line diagram (skeletal formula) of ethylene glycol:

![Line drawing of ethylene glycol](https://upload.wikimedia.org/wikipedia/commons/a/a6/Ethylene_glycol.svg)

In the 2D diagram the molecular structure appears linear, but let's use GOAT to find out the lowest-energy conformers of this flexible molecular structure.

Start by creating a new subdirectory:

In [ ]:
!mkdir ethylene_glycol

### Loading ethylene glycol from PubChem

Load the molecular structure from PubChem and visualize it:

In [ ]:
eg_view = load_molecule_pubchem(name = "Ethane-1,2-diol", xyzfile = f"ethylene_glycol/ethylene_glycol_pubchem.xyz")
show_molecule(eg_view)

The molecular structure loaded from PubChem already implies that the conformers should be carefully analyzed. There appears to be possibility for intramolecular hydrogen bonding.

### Conformational analysis with GOAT

Let's run the GOAT algorithm to get an estimate of low-energy conformers. GOAT will run a vast number of structure optimization calculations to search for low-energy conformers (consider that the C-C bond can rotate freely and so can the OH groups).

We will use fast, semiempirical XTB-type method by Grimme and coworkers (extended tight-binding, GFN2-xTYB method). These semiempirical methods have been derived from DFT but empirical parametrization enables orders of magnitude faster calculations than true DFT without losing too much accuracy.

In [ ]:
%%writefile ethylene_glycol/ethylene_glycol_goat.inp
!GFN2-xTB GOAT
!PAL4
* xyzfile 0 1 ethylene_glycol_pubchem.xyz

The job will take about 4 to 5 minutes to complete:

In [ ]:
run_orca(jobdir = "ethylene_glycol", jobname = "ethylene_glycol_goat")

After the job has completed, print the results of the analysis (*conformational ensemble*):

In [ ]:
!sed -n -e '/Final ensemble info/,/Conformers below/p' ethylene_glycol/ethylene_glycol_goat.out

Note that GOAT uses a stochastic approach and the number of conformers may vary slightly from run to run. ORCA reports how many of the conformers are within 3 kcal/mol (16.7 kJ/mol) from each other. There should be about 10 in the case of ethylene glycol. 

Let's visualize all conformers (change the frames to see all conformers):

In [ ]:
conformers = load_xyz_as_traj("ethylene_glycol/ethylene_glycol_goat.finalensemble.xyz")
show_molecule(conformers)

### Optimizing the conformers with DFT

The XTB-type methods typically produce a good initial guess of the conformers. The follow-up calculations are then carried out with DFT.

The number of conformers that will be treated with DFT depends on the system and the research question. In this case, we are just interested in finding the lowest-energy conformer. Let's optimize all conformers produced by GOAT using DFT-PBE-D3/def2-SVP level of theory (def2-SVP is a smaller basis set than the def2-TZVP we have used before).

In [ ]:
%%writefile ethylene_glycol/ethylene_glycol_conf.inp
!PBE D3 DEF2-SVP Opt
!PAL4
* xyzfile 0 1 ethylene_glycol_goat.finalensemble.xyz

Run the job (will take 10 to 13 minutes)

In [ ]:
run_orca(jobdir = "ethylene_glycol", jobname = "ethylene_glycol_conf")

Let's plot the relative energies of the conformers:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energies_raw = !cat ethylene_glycol/ethylene_glycol_conf.xyzact.dat | awk '{print $NF}'
energies = np.asarray(energies_raw, dtype=float)
rel_energies = (energies - np.min(energies)) * HARTREE_TO_KJMOL
conf_num = np.arange(1, len(energies_raw) + 1)

plt.plot(conf_num, rel_energies, marker = 'o', color = 'red', label = 'Relative energy')
plt.xlabel('Conformer number')
plt.ylabel('Relative energy (kJ/mol)')
plt.xlim(1, len(conf_num) + 1)
plt.ylim(0, np.max(rel_energies) * 1.05)
plt.xticks(np.arange(1, len(conf_num) + 1, 1))
plt.legend(loc = 'upper left')
plt.title('Ethylene glycol conformers')
plt.show()

The conformers are numbered by their original XTB energy ordering. Note how the relative energies of the conformers changed during the DFT optimization. Some of the conformers can fall to the same local minimum during the DFT optimization (their relative energies become practically identical).

Let's visualize the optimized conformers (small editing of the XYZ file with `sed` is needed before that).

In [ ]:
!sed '/^>/d' ethylene_glycol/ethylene_glycol_conf.allxyz > ethylene_glycol/ethylene_glycol_conf.all.xyz

In [ ]:
conformers_dft = load_xyz_as_traj("ethylene_glycol/ethylene_glycol_conf.all.xyz")
show_molecule(conformers_dft)

### Final calculation on the lowest-energy conformer

Finally, let's carry out an accurate DFT-PBE0/def2-TZVP optimization and frequency calculation on the lowest-energy conformer.

Fill in the number of the lowest-energy conformer in the ORCA input file below (replace `FIXME` with the conformer number, for example `008`).

In [ ]:
%%writefile ethylene_glycol/ethylene_glycol_pbe0.inp
!PBE0 D3 DEF2-TZVP Opt Freq
!PAL4
* xyzfile 0 1 ethylene_glycol_conf.001.xyz

Run the job (will take 4 to 5 minutes)

In [ ]:
run_orca(jobdir = "ethylene_glycol", jobname = "ethylene_glycol_pbe0")

Visualize the optimization trajectory and the final molecular structure of the lowest-energy conformer:

In [ ]:
pbe0_opt = load_xyz_as_traj("ethylene_glycol/ethylene_glycol_pbe0_trj.xyz")
show_molecule(pbe0_opt)

Let's also generate IR spectrum and plot it.

*Technical note:* In the code below, we scale the harmonic frequencies by a constant factor of `0.97`. This is rather typical procedure in the literature to facilitate comparisons with experimental spectra. The scaling is empirical and depends on the level of theory. It can also be considered to account for the lack of anharmonicity in the vibrational modes.

In [ ]:
!cd ethylene_glycol ; orca_mapspc ethylene_glycol_pbe0.hess IR -w32

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# Scale the harmonic frequencies by a constant factor for easier comparison with experiment.
# Rationale: (1) lack of anharmonicity in the calculation, (2) DFT is far from perfect.
FREQ_SCALE = 0.97 

# Load the raw data
ir_data = np.loadtxt('ethylene_glycol/ethylene_glycol_pbe0.hess.ir.dat')
frequencies = ir_data[:, 0] * FREQ_SCALE # Wavenumbers in the first column
transmittances = (ir_data[:, 1] - 999) * 100 # Transmittances in the second column
plt.plot(frequencies, transmittances, '-', color = 'red', label = 'IR')
plt.xlabel('Frequency (cm$^{-1}$)')
plt.ylabel('Transmittance (%)')
plt.xlim(max(frequencies), min(frequencies), )
plt.ylim(min(transmittances) * 0.999, max(transmittances) * 1.001)
plt.xticks(np.linspace(500, 4000, 8))
plt.yticks(np.linspace(min(transmittances), max(transmittances), 10))
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
plt.title('DFT-PBE0/def2-TZVP IR spectrum of ethylene glycol')
plt.show()

Compare the calculated spectrum with the experimental one (NIST WebBook):


<img src="https://webbook.nist.gov/cgi/cbook.cgi?Spec=C107211&Index=0&Type=IR&Large=on" alt="Ethylene glycol gas-phase IR spectrum" width="600" height="400">

> #### Question
> How well do the calculated and experimental spectrum agree?

Write your answer here: 

## Part 5. Note about point group symmetry

Point group symmetry can be used to speed up quantum chemical calculations. ORCA can utilize the full point group of a molecule in structural optimizations (less degrees of freedom to optimize). In the actual electronic structure calculation, it can use at most point group $D_{2h}$ and its subgroups. Only a few quantum chemical program packages utilize the full point group in electronic structure calculations (for example, TURBOMOLE).

Previously, we optimized the structure of WF<sub>6</sub> and calculated harmonic frequencies for it "without symmetry" (point group *C*<sub>1</sub>). In reality, the point group of the molecule is *O<sub>h</sub>* which means that it possesses 48 symmetry operations.

Write an input file for WF<sub>6</sub> with the keyword `UseSymmetry`. Note that `UseSymmetry` will reorient the molecule in such way that it will be compatible with the requirements of the point group.

In [ ]:
%%writefile wf6/wf6_oh.inp
!PBE0 DEF2-TZVP Opt Freq UseSymmetry
!PAL4
* xyz 0 1
 W                 -0.13573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664    1.87644611
 F                  1.74426503    0.20359664   -0.00355389
 F                 -2.01573497    0.20359664   -0.00355389
 F                 -0.13573497    0.20359664   -1.88355389
 F                 -0.13573497    2.08359664   -0.00355389
 F                 -0.13573497   -1.67640336   -0.00355389
*

Run the job:

In [ ]:
run_orca(jobdir = "wf6", jobname = "wf6_oh")

Print the vibrational frequencies:

In [ ]:
!sed -n -e '/VIBRATIONAL FREQUENCIES/,/NORMAL MODES/p' wf6/wf6.out | head -n -3

> #### Question
> Did the job run any faster? Did the vibrational frequencies change significantly?

## The End

This concludes the basic Orca tutorial.